# 第3回　ばらつきを測る／正規分布という「仮定」

統計学Ⅰ（B）　／　北星学園大学

今日も**▶を上から押すだけ**。注目するのは ――

> 平均が同じでも、**ばらつき**が違えば、まったく別のデータだ。

そして後半では、前回の続きをやる。**「正規分布だと思ってよいか」は、データだけでは決まらない。**

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## フック：平均が同じ2つの調査地

2か所の森で、サルの体重を30頭ずつ測った。**どちらも平均6.0kg**。
あなたは「同じような群れだ」と思うだろうか？

- 森A：ほとんどの個体が6kg前後
- 森B：半分が2kg、半分が10kg

In [ ]:
森A = np.full(30, 6.0)                        # ほぼ全員が平均
森B = np.array([2.0]*15 + [10.0]*15)          # 二極化

print(f"森A  平均 {森A.mean():.1f}kg  標準偏差 {森A.std(ddof=1):.2f}")
print(f"森B  平均 {森B.mean():.1f}kg  標準偏差 {森B.std(ddof=1):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
ax[0].hist(森A, bins=np.arange(0,13,0.5), color="#80cbc4", edgecolor="white")
ax[0].set_title("森A（平均6.0kg）")
ax[1].hist(森B, bins=np.arange(0,13,0.5), color="#e8503a", edgecolor="white")
ax[1].set_title("森B（平均6.0kg）")
for a in ax:
    a.set_xlabel("体重(kg)"); a.axvline(6.0, color="gray", ls="--")
plt.show()

**平均は同じ6.0kg。でも中身は正反対。** 平均だけでは、この違いは絶対に見えない。

森Bは、そもそも**2つの別々の集団が混ざっている**のかもしれない（前回の二山の話とつながる）。
足りないのは「**どれだけ散らばっているか**」＝ばらつきの情報だ。

---
## ばらつきを測る：分散と標準偏差

ばらつきは「各データが平均からどれだけ離れているか」で測る。

1. 各データの **平均からの差**（偏差）を出す
2. そのままだとプラスとマイナスで打ち消し合う → **二乗**してから平均する ＝ **分散**
3. 二乗したままだと単位が日²で直感的でない → **ルートを取って元の単位に戻す** ＝ **標準偏差(SD)**

霊長類の `妊娠期間日` で計算してみよう。

In [ ]:
g = df["妊娠期間日"].dropna()

print(f"種数　　　 {len(g)} 種")
print(f"平均　　　 {g.mean():.1f} 日")
print(f"分散　　　 {g.var():.1f} （日の二乗・直感的でない）")
print(f"標準偏差SD {g.std():.1f} 日（元の単位に戻った・これを使う）")

plt.figure(figsize=(8,4))
plt.hist(g, bins=25, color="#80cbc4", edgecolor="white")
m, s = g.mean(), g.std()
plt.axvline(m, color="#1565c0", lw=2, label=f"平均 {m:.0f}日")
plt.axvspan(m-s, m+s, color="#e8503a", alpha=0.15, label=f"平均±1SD（{m-s:.0f}〜{m+s:.0f}日）")
plt.xlabel("妊娠期間（日）"); plt.ylabel("種数"); plt.legend()
plt.title("標準偏差＝平均からの『標準的な散らばり幅』")
plt.show()

> **平均は分布の『位置』、SDは『幅』。** ヒトの妊娠期間は約280日で、この平均164日から見ると **+3SD** ほど右にいる。霊長類の中ではかなり長いほうである。

---
## 偏差値の正体 ―― そして、それが壊れるとき

「偏差値」は、値を **平均50・標準偏差10** のものさしに置き直したもの。

$$ 偏差値 = 50 + 10 \times \frac{あなたの値 - 平均}{標準偏差} $$

つまり「平均からSD何個分ずれているか」を 50 中心に表しただけ。**これを霊長類の体重でやってみよう。**

In [ ]:
w = df["体重g"].dropna()
m, s = w.mean(), w.std()

def 偏差値(値, 平均, 標準偏差):
    return 50 + 10 * (値 - 平均) / 標準偏差

print(f"（体重の平均 {m:,.0f}g・SD {s:,.0f}g で計算）\n")
for sp in ["Microcebus myoxinus", "Macaca fuscata", "Pan troglodytes",
           "Homo sapiens", "Gorilla beringei"]:
    r = df[df["学名"] == sp]
    v = r["体重g"].iloc[0]
    print(f"  {sp:<22} {v:>9,.0f} g  → 偏差値 {偏差値(v, m, s):6.1f}")

### 上は青天井、下は潰れる

**ゴリラの偏差値が 159。** まともなテストではありえない数字である。

そして、もう一方の端も見てほしい。**31gのネズミキツネザルが偏差値 45.5。**10kgのニホンザル（53.2）と、たった8しか違わない。**体重は324倍も違うのに**である。

> 上側では1種が159まで飛び出し、下側では300倍の差が8ポイントに潰れている。

**計算式は間違っていない。**壊れているのは、**この式が前提にしているもの**のほうである。

偏差値は、**データが平均のまわりに左右対称に散らばっている**ことを暗黙に仮定している。前回見たとおり、体重の分布は右に長い裾を引いていた。そこへ「平均からSD何個分」というものさしを当てると、**裾にいる種の値が青天井になる。**

前回の最後にやったことを思い出そう。**目盛りを変える。**

In [ ]:
lw = np.log10(w)
m2, s2 = lw.mean(), lw.std()

print("　　　　　　　　　　　　　　　  生のスケール   対数スケール")
for sp in ["Microcebus myoxinus", "Macaca fuscata", "Pan troglodytes",
           "Homo sapiens", "Gorilla beringei"]:
    r = df[df["学名"] == sp]
    v = r["体重g"].iloc[0]
    a = 偏差値(v, m, s)
    b = 偏差値(np.log10(v), m2, s2)
    print(f"  {sp:<22} {a:>8.1f}    {b:>8.1f}")

**対数スケールなら、全部 23〜78 の範囲に収まった。** 常識的な偏差値の幅である。

> **同じ種の、同じ体重である。変えたのはものさしだけ。**
> 「ゴリラは偏差値159の巨体」と言うことも、「偏差値77」と言うこともできる。
> **どちらの数字を出すかで、読む人の受ける印象はまったく変わる。**

偏差値・Zスコア・標準化――名前は違っても、すべて同じ仮定の上に乗っている。**その仮定が成り立っているかを確かめずに使うと、こうなる。**

---
## 正規分布と 68-95-99.7 則

左右対称で釣鐘型の分布を **正規分布** という。正規分布なら、ばらつきの目安が決まっている：

- 平均 ±1SD に約 **68%**
- 平均 ±2SD に約 **95%**
- 平均 ±3SD に約 **99.7%**

**対数スケールの体重**が、本当にこの通りか数えてみよう。

In [ ]:
m2, s2 = lw.mean(), lw.std()
print("【対数スケールの体重】")
for k, 理論 in [(1,68),(2,95),(3,99.7)]:
    実際 = ((lw >= m2-k*s2) & (lw <= m2+k*s2)).mean()*100
    print(f"  平均±{k}SD に 実際 {実際:5.1f}%  （理論 {理論}%）")

x = np.linspace(lw.min(), lw.max(), 200)
plt.figure(figsize=(8,4))
plt.hist(lw, bins=30, density=True, color="#80cbc4", edgecolor="white", label="実データ")
plt.plot(x, stats.norm.pdf(x, m2, s2), color="#e8503a", lw=2, label="正規分布（平均とSDから）")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.xlabel("体重（対数目盛り）"); plt.ylabel("割合"); plt.legend()
plt.title(f"対数にすると正規分布でだいたい近似できる（歪度 {stats.skew(lw):.2f}）")
plt.show()

**72% / 95% / 100%。** 理論の 68/95/99.7 に近い。悪くない近似である。

では、**生のスケール**ではどうなるか。

In [ ]:
x = np.linspace(0, 40000, 300)
plt.figure(figsize=(8,4))
plt.hist(w[w < 40000], bins=40, density=True, color="#80cbc4",
         edgecolor="white", label="実データ（体重）")
plt.plot(x, stats.norm.pdf(x, m, s), color="#e8503a", lw=2, label="正規分布を当てはめると…")
plt.xlabel("体重（g）"); plt.ylabel("割合"); plt.legend()
plt.title(f"生のスケールでは正規分布で表せない（歪度 {stats.skew(w):.1f}・右に裾）")
plt.show()

print("【生のスケールの体重】")
for k, 理論 in [(1,68),(2,95),(3,99.7)]:
    実際 = ((w >= m-k*s) & (w <= m+k*s)).mean()*100
    print(f"  平均±{k}SD に 実際 {実際:5.1f}%  （理論 {理論}%）")
print()
print(f"平均 − 1SD = {m-s:,.0f} g   ← 体重がマイナスの霊長類はいない")

赤い正規分布の曲線は、データにまるで合っていない。
しかも **平均−1SD が負の値**になる。**体重がマイナスの動物はいない。**
正規分布を当てはめた時点で、ありえない領域に確率を配ってしまっている。

---
## 正直に言うと、対数にしても「正規分布」ではない

ここまで「対数にすればうまくいく」と話してきたが、**厳密に検定すると、どちらも正規分布ではない**。

In [ ]:
for name, v in [("生のスケール", w), ("対数スケール", lw), ("妊娠期間日", g)]:
    sh = stats.shapiro(v)
    print(f"{name:<12} 歪度 {stats.skew(v):+6.2f}  尖度 {stats.kurtosis(v):+6.2f}  "
          f"正規性検定 p={sh.pvalue:.4f} → {'正規とは言えない' if sh.pvalue < 0.05 else '正規を否定できない'}")

**3つとも「正規分布とは言えない」と出た。** 妊娠期間日は歪度がほぼ0（−0.06）で見た目は左右対称なのに、である（尖度が高く、山が尖りすぎている）。

> **左右対称であることと、正規分布であることは、同じではない。**

では対数変換は無駄だったのか。そうではない。**±1SDの実測が 72%（理論68%）まで近づいた**のは事実で、生のスケールの状態よりはるかに使える。

> **正規分布は「現実の姿」ではなく「近似の道具」である。**
> 問うべきは「正規分布か（Yes/No）」ではなく、**「この目的に対して、十分な近似か」**。
> そして、それを決めるのは検定ではなく、**何に使うかを知っているあなた**である。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 分散 | 平均からの差を二乗して平均（単位は元の二乗） |
| 標準偏差(SD) | 分散のルート。**ばらつきの標準的な幅**（元の単位） |
| 偏差値 | 平均50・SD10 に置き直した相対位置。**分布が歪むと破綻する**（ゴリラ159） |
| 68-95-99.7則 | 正規分布なら±1/2/3SDに68/95/99.7%（**正規のときだけ**） |

> **平均は分布の『位置』、SDは『幅』。両方見て初めてデータが分かる。**
> そして正規分布は便利な仮定だが、当てはまるかは**自分で確かめる**もの。

今日いちばん覚えて帰ってほしいのは、これである。

> **同じデータでも、ものさしを変えれば偏差値は159にも77にもなる。**
> 数字が出てきたら、**どんな仮定の上で計算されたのか**を問う。

**課題（Moodle）**：平均が同じでSDが違う2データの解釈／偏差値の読み方。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。